In [1]:
T = CartanType(['A', 4])
W = WeylGroup(T,prefix="s")
[s1, s2, s3, s4] = W.simple_reflections()

WCR = WeylCharacterRing(T, style="coroots", base_ring=QQ)
R = WeightRing(WCR)
# EXPI MODIFIED, THIS CODE ONLY WORKS IN TYPE A
f = [w.coerce_to_sl() for w in WCR.fundamental_weights()]

A4 = WCR

n = len(W.simple_reflections())
e = s1^2
w0 = W.long_element()
S.<q> = LaurentPolynomialRing(QQ)
KL = KazhdanLusztigPolynomial(W,q)
q = 7

def Iw(w):
    a = []
    for i in range(1, n+1):
        if w.has_right_descent(i):
            a.append(i)
    return a

def Iwc(w):
    a = []
    for i in range(1, n+1):
        if not w.has_right_descent(i):
            #a.append(W.simple_reflections()[i])
            a.append(i)
    return a

def expI(l):
    a = R.one()
    for i in l:
        a = a * R(f[i-1])
    return a

def Cpsub_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(y)
        s += int(KL.P(y, w)(1)) * a
    return s

def Cpsup_on_wr_element(w, v):
    interv = W.bruhat_interval(e, w0*w)
    s = 0
    for y in interv:
        a = v.weyl_group_action(w0*y)
        s += int(KL.P(y, w0*w)(1)) * a
    return s

def fsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)))

def qfsub(w):
    return Cpsub_on_wr_element(w, expI(Iwc(w)).scale(q))

def fsup(w):
    return Cpsup_on_wr_element(w, expI(Iw(w)))

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

def Iweight(w):
    wgt = 0
    for a in Iw(w):
        wgt += f[a-1]
    return wgt

rho = Iweight(w0)

def weightpairing(l):
    a = WCR.dot_reduce(l - rho)
    if a[0] == 0:
        return 0
    else:
        return a[0]*WCR(a[1])

def woke_pairing(a, b):
    #takes two elements of R and gives you an element of WCR
    d = dict(R(a*b))
    r = 0
    for l in d:
        r += d[l]*weightpairing(l)
    return r

invols = []
for w in W:
    if w*w == e:
        invols.append(w)

d_sub = {}
d_sup = {}

for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_sub[i] = fsub(W[i])
    d_sup[i] = fsup(W[i])

print("Setup done, f_w, f^w are defined.")

Setup done, f_w, f^w are defined.


In [2]:
def build_matrix():
    m = []
    for i in range(len(W)):
        print(i + 1, "out of", len(W), end='\r')
        r = []
        for j in range(len(W)):
            r.append(woke_pairing(d_sub[i], d_sup[j]))
        m.append(r)
    return m
bigmat = build_matrix()
#bigmat[i][j] = woke_pairing(d_sub[i], d_sup[j]) for all i, j
alls = list(range(len(W)))
pops = list(range(len(W)))
order = []
while len(pops) > 0:
    for a in pops:
        good = True
        for b in pops:
            if b != a and bigmat[b][a] != 0:
                good = False
        if good:
            order.append(a)
            pops.remove(a)

order.reverse()
print("Computed the matrix of pairings and the upper triangular order.")

Computed the matrix of pairings and the upper triangular order.


In [3]:
def lc(i):
    a = bigmat[i][i]
    d = dict(WCR(a))
    if len(d) != 1:
        print("LC ERROR")
    else:
        return list(d.values())[0]

counter = 0
d_dual = {}
for j in order:
    counter += 1
    print(counter, "out of", len(W), end='\r')
    d_dual[j] = d_sup[j]/lc(j)
    for i in order:
        if  i != j and woke_pairing(d_sub[i], d_dual[j]) != 0:
            d_dual[j] = d_dual[j] - woke_pairing(d_sub[i], d_dual[j])*d_sup[i]/lc(i)
print("Computed the dual basis as the dictionary d_dual.")

Computed the dual basis as the dictionary d_dual.


In [ ]:
#OPTIONAL CELL: CHECKING THE DUAL BASIS IS VALID.
valid = True
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    for j in range(len(W)):
        a = woke_pairing(d_sub[i], d_dual[j])
        if a != 0:
            if i != j or a != 1:
                print("ISSUE AT", i, j)
                valid = False
if valid:
    print("Checked dual basis, all is good.")
else:
    print("ISSUES! Dual basis is NOT correct.")

In [ ]:
#SKIP TO THIS PART, DUAL ALREADY DEFINED
d_check = {84: 1/24*R(0,-1,-1,-1) + 1/24*R(0,-1,-2,1) + 1/24*R(0,-2,1,-2) + 1/12*R(0,-2,0,0) + 1/24*R(0,-2,-1,2) + 1/24*R(-1,1,-2,-1) + 1/24*R(-1,1,-3,1) + 1/12*R(-1,0,0,-2) + 3/8*R(-1,0,-1,0) + 1/12*R(-1,0,-2,2) + 1/24*R(-1,-1,2,-3) + 5/24*R(-1,-1,1,-1) + 5/24*R(-1,-1,0,1) + 1/24*R(-1,-1,-1,3) - 1/24*R(-2,1,0,-1) - 1/24*R(-2,1,-1,1) + 1/8*R(1,-1,-1,0) + 1/12*R(1,-2,1,-1) + 1/12*R(1,-2,0,1) + 1/6*R(0,1,-2,0) + 1/4*R(0,0,0,-1) + 1/4*R(0,0,-1,1) + 1/12*R(0,-1,2,-2) + 1/8*R(0,-1,1,0) + 1/12*R(0,-1,0,2) - 1/24*R(-1,2,-1,-1) - 1/24*R(-1,2,-2,1) - 1/24*R(-1,1,1,-2) - 1/12*R(-1,1,0,0) - 1/24*R(-1,1,-1,2), 117: 1/24*R(-1,-1,-1,0) + 1/24*R(-1,-2,1,-1) + 1/6*R(0,-1,0,-1) + 1/12*R(0,-1,-1,1) + 1/12*R(0,-2,1,0) - 1/24*R(-1,1,-1,-1) - 1/24*R(-1,1,-2,1) - 1/24*R(-1,0,1,-2) - 1/24*R(-1,-1,2,-1) + 1/8*R(1,-1,0,0), 47: 1/24*R(-1,-1,-1,0) + 1/24*R(-1,-2,1,-1) + 1/24*R(-2,1,-2,0) + 1/12*R(-2,0,0,-1) + 1/24*R(-3,2,-1,-1) + 1/24*R(1,-2,-1,0) + 1/24*R(1,-3,1,-1) + 1/12*R(0,0,-2,0) + 3/8*R(0,-1,0,-1) + 1/8*R(0,-1,-1,1) + 1/6*R(0,-2,1,0) + 5/24*R(-1,1,-1,-1) + 1/12*R(-1,1,-2,1) - 1/24*R(-1,0,1,-2) + 1/4*R(-1,0,0,0) - 1/24*R(-1,-1,2,-1) + 1/12*R(-2,2,-1,0) - 1/24*R(-2,1,1,-1) + 1/24*R(2,-1,-2,0) + 1/12*R(2,-2,0,-1) + 5/24*R(1,0,-1,-1) + 1/12*R(1,0,-2,1) - 1/24*R(1,-1,1,-2) + 1/4*R(1,-1,0,0) - 1/24*R(1,-2,2,-1) + 1/8*R(0,1,-1,0) - 1/12*R(0,0,1,-1) + 1/24*R(3,-1,-1,-1) + 1/12*R(2,0,-1,0) - 1/24*R(2,-1,1,-1), 99: 1/24*R(0,-1,-1,-1) + 1/24*R(-1,1,-2,-1) + 1/6*R(-1,0,-1,0) - 1/24*R(-1,-1,1,-1) - 1/24*R(-2,1,0,-1) + 1/12*R(1,-1,-1,0) - 1/24*R(1,-2,1,-1) + 1/12*R(0,1,-2,0) + 1/8*R(0,0,-1,1) - 1/24*R(-1,2,-1,-1), 61: 3/4*R(0,-1,0,-1) + 3/4*R(0,-1,-1,1) + 1/2*R(0,-2,1,0) + 3/4*R(-1,1,-1,-1) + 3/4*R(-1,1,-2,1) + 1/2*R(-1,0,1,-2) + 9/4*R(-1,0,0,0) + 1/2*R(-1,0,-1,2) + 1/4*R(-1,-1,2,-1) + 1/4*R(-1,-1,1,1) + 1/2*R(-2,2,-1,0) + 1/4*R(-2,1,1,-1) + 1/4*R(-2,1,0,1) + 1/2*R(1,0,-1,-1) + 1/2*R(1,0,-2,1) + 1/4*R(1,-1,1,-2) + 3/2*R(1,-1,0,0) + 1/4*R(1,-1,-1,2) + 1/4*R(1,-2,2,-1) + 1/4*R(1,-2,1,1) + 1/4*R(0,1,0,-2) + 3/2*R(0,1,-1,0) + 1/4*R(0,1,-2,2) + 3/4*R(0,0,1,-1) + 3/4*R(0,0,0,1) + 1/4*R(-1,2,0,-1) + 1/4*R(-1,2,-1,1), 25: -1/4*R(-1,0,-1,0) - 1/4*R(-1,-1,1,-1) - 1/4*R(-1,-1,0,1) - 1/4*R(1,-1,-1,0) - 1/4*R(1,-2,1,-1) - 1/4*R(1,-2,0,1) - 1/4*R(0,0,0,-1) - 1/4*R(0,0,-1,1) - 1/4*R(0,-1,1,0), 107: -1/24*R(-1,-1,-1,0) - 1/24*R(-1,-2,1,-1) - 1/24*R(-2,1,-2,0) - 1/12*R(-2,0,0,-1) - 1/24*R(-3,2,-1,-1) - 1/6*R(0,-1,0,-1) - 1/24*R(0,-1,-1,1) - 1/12*R(0,-2,1,0) - 1/6*R(-1,1,-1,-1) - 1/24*R(-1,1,-2,1) + 1/24*R(-1,0,1,-2) - 5/24*R(-1,0,0,0) + 1/24*R(-1,-1,2,-1) - 1/12*R(-2,2,-1,0) + 1/24*R(-2,1,1,-1) - 1/12*R(1,-1,0,0) - 1/12*R(0,1,-1,0) + 1/24*R(0,0,1,-1), 91: -1/2*R(0,-1,0,-1) - 1/4*R(0,-1,-1,1) - 1/4*R(0,-2,1,0) - 1/2*R(-1,1,-1,-1) - 1/4*R(-1,1,-2,1) - 1/2*R(-1,0,1,-2) - R(-1,0,0,0) - 1/4*R(-1,-1,2,-1) - 1/4*R(-2,2,-1,0) - 1/4*R(-2,1,1,-1) - 1/4*R(1,0,-1,-1) - 1/4*R(1,0,-2,1) - 1/4*R(1,-1,1,-2) - 3/4*R(1,-1,0,0) - 1/4*R(1,-2,2,-1) - 1/4*R(0,1,0,-2) - 3/4*R(0,1,-1,0) - 3/4*R(0,0,1,-1) - 1/4*R(-1,2,0,-1), 72: 1/6*R(0,-2,0,1) + 1/6*R(0,-2,1,-1) + 1/3*R(0,-1,-1,0) + 1/3*R(-2,1,0,0) + 1/6*R(-2,2,-2,1) + 1/6*R(-2,2,-1,-1) + 1/6*R(-2,1,1,-2) + 1/6*R(-2,1,-1,2) + 1/3*R(-1,1,-2,0) + 1/3*R(-1,-1,1,0) + 1/6*R(-1,-1,2,-2) + 1/6*R(-1,-1,0,2) + 2/3*R(-1,0,-1,1) + 2/3*R(-1,0,0,-1), 51: 1/2*R(0,-1,0,-1) + 1/4*R(0,-1,-1,1) + 1/4*R(0,-2,1,0) + 1/4*R(-1,1,-1,-1) + 1/4*R(-1,1,-2,1) + 1/4*R(-1,0,1,-2) + 1/2*R(-1,0,0,0) + 1/4*R(-1,-1,2,-1) + 1/4*R(1,0,-1,-1) + 1/4*R(1,0,-2,1) + 1/4*R(1,-1,1,-2) + 1/2*R(1,-1,0,0) + 1/4*R(1,-2,2,-1) + 1/4*R(0,1,-1,0) + 1/4*R(0,0,1,-1), 37: -1/6*R(0,-2,1,-1) - 1/6*R(0,-1,-1,0) - 1/6*R(-2,2,-1,-1) - 1/6*R(-1,1,-2,0) - 1/3*R(-1,0,0,-1) - 1/6*R(2,0,-1,-1) - 1/6*R(1,0,-2,0) - 1/3*R(1,-1,0,-1) - 1/3*R(0,1,-1,-1), 12: -5/6*R(0,0,0,0) - 1/3*R(0,-2,0,1) - 1/3*R(0,-2,1,-1) - 2/3*R(0,-1,-1,0) - 1/3*R(-2,1,0,0) - 1/6*R(-2,2,-2,1) - 1/6*R(-2,2,-1,-1) - 1/6*R(-2,1,1,-2) - 1/6*R(-2,1,-1,2) - 1/3*R(-1,1,-2,0) - 1/3*R(-1,-1,1,0) - 1/6*R(-1,-1,2,-2) - 1/6*R(-1,-1,0,2) - R(-1,0,-1,1) - R(-1,0,0,-1) - 1/3*R(2,-1,0,0) - 1/6*R(2,0,-2,1) - 1/6*R(2,0,-1,-1) - 1/6*R(2,-1,1,-2) - 1/6*R(2,-1,-1,2) - 1/3*R(1,0,-2,0) - 1/3*R(1,-2,1,0) - 1/6*R(1,-2,2,-2) - 1/6*R(1,-2,0,2) - R(1,-1,-1,1) - R(1,-1,0,-1) - 1/3*R(0,1,-2,1) - 1/3*R(0,1,-1,-1) - 1/3*R(0,0,1,-2) - 1/3*R(0,0,-1,2), 119: 1/120*R(-1,-1,-1,-1) - 1/120*R(-2,-1,1,-1) + 1/60*R(-2,0,-1,0) + 1/30*R(0,-1,-1,0) + 1/60*R(0,-1,0,-2) - 1/120*R(-1,1,-1,-2) - 1/120*R(-1,-1,1,0) + 1/120*R(-1,0,-1,1) + 1/60*R(-1,0,0,-1) + 1/120*R(1,-1,-1,1) + 1/120*R(1,-1,0,-1) - 1/120*R(0,1,-1,-1), 113: 1/6*R(-1,-1,0,-1) + 1/12*R(-1,-1,-1,1) + 1/12*R(-1,-2,1,0) + 1/12*R(-2,1,-1,-1) + 1/12*R(-2,1,-2,1) + 1/12*R(-2,0,1,-2) + 1/4*R(-2,0,0,0) + 1/12*R(-2,-1,2,-1) + 1/6*R(0,-1,0,0) + 1/12*R(-1,1,-1,0) + 1/12*R(-1,0,1,-1), 102: 1/6*R(0,0,0,0) - 1/6*R(0,-2,1,-1) - 1/6*R(0,-1,-1,0) - 1/6*R(-2,2,-1,-1) - 1/6*R(-2,1,1,-2) - 1/6*R(-1,1,-2,0) - 1/6*R(-1,-1,2,-2) - 1/3*R(-1,0,0,-1), 93: 1/4*R(-1,-1,0,-1) + 1/6*R(-1,-1,-1,1) + 1/12*R(-1,-2,1,0) + 1/4*R(-2,1,-1,-1) + 1/6*R(-2,1,-2,1) + 1/6*R(-2,0,1,-2) + 1/2*R(-2,0,0,0) + 1/12*R(-2,-1,2,-1) + 1/12*R(-3,2,-1,0) + 1/12*R(-3,1,1,-1) + 1/6*R(1,-2,0,-1) + 1/12*R(1,-2,-1,1) + 1/12*R(1,-3,1,0) + 1/2*R(0,0,-1,-1) + 1/3*R(0,0,-2,1) + 1/3*R(0,-1,1,-2) + 11/12*R(0,-1,0,0) + 1/6*R(0,-2,2,-1) + 1/6*R(-1,2,-2,-1) + 1/12*R(-1,2,-3,1) + 1/3*R(-1,1,0,-2) + 11/12*R(-1,1,-1,0) + 2/3*R(-1,0,1,-1) + 1/12*R(-2,3,-2,0) + 1/6*R(-2,2,0,-1) + 1/12*R(2,-1,-1,-1) + 1/12*R(2,-1,-2,1) + 1/12*R(2,-2,1,-2) + 1/4*R(2,-2,0,0) + 1/12*R(2,-3,2,-1) + 1/12*R(1,1,-2,-1) + 1/12*R(1,1,-3,1) + 1/6*R(1,0,0,-2) + 2/3*R(1,0,-1,0) + 5/12*R(1,-1,1,-1) + 1/12*R(0,2,-1,-2) + 1/4*R(0,2,-2,0) + 5/12*R(0,1,0,-1) + 1/12*R(-1,3,-1,-1), 89: -1/24*R(-1,-1,-1,0) - 1/12*R(0,-1,0,-1) - 1/12*R(0,-1,-1,1) + 1/24*R(-1,1,-1,-1) + 1/24*R(-1,1,-2,1), 80: -1/6*R(0,0,-1,-1) - 1/6*R(0,0,-2,1) - 1/6*R(0,-1,1,-2) - 1/3*R(0,-1,0,0) - 1/6*R(0,-1,-1,2) - 1/6*R(0,-2,2,-1) - 1/6*R(0,-2,1,1) - 1/6*R(-1,1,0,-2) - 1/3*R(-1,1,-1,0) - 1/6*R(-1,1,-2,2) - 1/3*R(-1,0,1,-1) - 1/3*R(-1,0,0,1) - 1/6*R(-1,-1,2,0), 68: 1/4*R(0,-1,0,-1) + 1/4*R(0,-1,-1,1) + 1/4*R(-1,1,-1,-1) + 1/4*R(-1,1,-2,1), 58: 1/4*R(0,-1,0,-1), 48: -1/12*R(-1,-1,0,-1) - 1/12*R(-2,1,-1,-1) - 1/12*R(1,-2,0,-1) - 1/6*R(0,0,-1,-1) - 1/3*R(0,-1,0,0) - 1/6*R(-1,1,-1,0) + 1/12*R(-1,0,1,-1) - 1/12*R(2,-1,-1,-1) - 1/6*R(1,0,-1,0) + 1/12*R(1,-1,1,-1), 42: 1/3*R(0,-2,1,-1) + 1/3*R(0,-1,-1,0) + 1/6*R(-2,2,-1,-1) + 1/6*R(-2,1,1,-2) + 1/6*R(-1,1,-2,0) + 1/6*R(-1,-1,2,-2) + 2/3*R(-1,0,0,-1) + 1/6*R(2,0,-1,-1) + 1/6*R(2,-1,1,-2) + 1/6*R(1,0,-2,0) + 1/6*R(1,-2,2,-2) + 2/3*R(1,-1,0,-1) + 1/3*R(0,1,-1,-1) + 1/3*R(0,0,1,-2), 30: -1/2*R(-1,0,0,0) - 1/2*R(1,-1,0,0) - 1/2*R(0,1,-1,0) - 1/2*R(0,0,1,-1), 18: 1/12*R(-1,-1,0,-1) + 1/12*R(-1,-1,-1,1) + 1/12*R(-2,1,-1,-1) + 1/12*R(-2,1,-2,1) + 1/12*R(-2,0,0,0) + 1/12*R(1,-2,0,-1) + 1/12*R(1,-2,-1,1) + 1/6*R(0,0,-1,-1) + 1/6*R(0,0,-2,1) + 1/3*R(0,-1,0,0) + 1/4*R(-1,1,-1,0) + 1/12*R(2,-1,-1,-1) + 1/12*R(2,-1,-2,1) + 1/12*R(2,-2,0,0) + 1/4*R(1,0,-1,0), 7: 1/2*R(0,-1,0,0) + 1/2*R(-1,1,-1,0) + 1/2*R(1,0,-1,0), 114: -1/24*R(0,-1,-1,-1) - 1/24*R(0,-2,1,-2) - 1/24*R(-1,1,-2,-1) - 1/12*R(-1,0,0,-2) - 1/6*R(-1,0,-1,0) - 1/24*R(-1,-1,2,-3) - 1/6*R(-1,-1,1,-1) + 1/24*R(-2,1,0,-1) - 1/24*R(1,-1,-1,0) - 1/24*R(1,-2,1,-1) - 1/12*R(0,1,-2,0) - 5/24*R(0,0,0,-1) - 1/12*R(0,0,-1,1) - 1/12*R(0,-1,2,-2) - 1/12*R(0,-1,1,0) + 1/24*R(-1,2,-1,-1) + 1/24*R(-1,1,1,-2) + 1/24*R(-1,1,0,0), 110: 1/4*R(-1,0,-1,-1) + 1/6*R(-1,0,-2,1) + 1/4*R(-1,-1,1,-2) + 1/2*R(-1,-1,0,0) + 1/12*R(-1,-1,-1,2) + 1/6*R(-1,-2,2,-1) + 1/12*R(-1,-2,1,1) + 1/6*R(-2,1,0,-2) + 1/3*R(-2,1,-1,0) + 1/12*R(-2,1,-2,2) + 1/3*R(-2,0,1,-1) + 1/6*R(-2,0,0,1) + 1/12*R(-2,-1,2,0) + 1/6*R(1,-1,-1,-1) + 1/12*R(1,-1,-2,1) + 1/6*R(1,-2,1,-2) + 1/3*R(1,-2,0,0) + 1/12*R(1,-2,-1,2) + 1/12*R(1,-3,2,-1) + 1/12*R(1,-3,1,1) + 1/12*R(0,1,-2,-1) + 1/12*R(0,1,-3,1) + 1/2*R(0,0,0,-2) + 11/12*R(0,0,-1,0) + 1/4*R(0,0,-2,2) + 1/12*R(0,-1,2,-3) + 11/12*R(0,-1,1,-1) + 2/3*R(0,-1,0,1) + 1/12*R(0,-2,3,-2) + 1/4*R(0,-2,2,0) + 1/12*R(-1,2,-1,-2) + 1/6*R(-1,2,-2,0) + 1/12*R(-1,2,-3,2) + 1/12*R(-1,1,1,-3) + 2/3*R(-1,1,0,-1) + 5/12*R(-1,1,-1,1) + 1/6*R(-1,0,2,-2) + 5/12*R(-1,0,1,0) + 1/12*R(-1,-1,3,-1), 106: -1/4*R(-1,0,0,-1), 96: 1/6*R(-1,0,-1,-1) + 1/12*R(-1,-1,1,-2) + 1/12*R(-2,1,0,-2) + 1/12*R(1,-1,-1,-1) + 1/12*R(1,-2,1,-2) + 1/12*R(0,1,-2,-1) + 1/4*R(0,0,0,-2) + 1/6*R(0,0,-1,0) + 1/12*R(0,-1,1,-1) + 1/12*R(-1,2,-1,-2) + 1/12*R(-1,1,0,-1), 86: -1/12*R(-1,0,-1,-1) - 1/12*R(-1,0,-2,1) - 1/12*R(-1,-1,1,-2) - 1/6*R(-1,-1,0,0) - 1/12*R(-1,-1,-1,2) - 1/3*R(0,0,-1,0) - 1/6*R(0,-1,1,-1) - 1/6*R(0,-1,0,1) + 1/12*R(-1,1,0,-1) + 1/12*R(-1,1,-1,1), 76: 1/4*R(-1,0,-1,1) + 1/4*R(-1,0,0,-1), 65: 1/2*R(-1,0,-1,0) + 1/4*R(-1,-1,1,-1) + 1/4*R(-1,-1,0,1) + 1/4*R(-2,1,0,-1) + 1/4*R(-2,1,-1,1) + 1/4*R(1,-1,-1,0) + 1/4*R(1,-2,1,-1) + 1/4*R(1,-2,0,1) + 1/4*R(0,1,-2,0) + 1/2*R(0,0,0,-1) + 1/2*R(0,0,-1,1) + 1/4*R(0,-1,1,0) + 1/4*R(-1,2,-1,-1) + 1/4*R(-1,2,-2,1) + 1/4*R(-1,1,0,0), 59: -1/24*R(0,-1,-1,-1) - 1/12*R(-1,0,-1,0) + 1/24*R(-1,-1,1,-1) - 1/12*R(1,-1,-1,0) + 1/24*R(1,-2,1,-1), 55: 1/4*R(-1,0,-1,0) + 1/4*R(-1,-1,1,-1) + 1/4*R(1,-1,-1,0) + 1/4*R(1,-2,1,-1), 46: 1/4*R(-1,0,0,-1) + 1/4*R(1,-1,0,-1), 40: 3/4*R(-1,0,-1,0) + 3/4*R(-1,-1,1,-1) + 1/2*R(-1,-1,0,1) + 1/2*R(-2,1,0,-1) + 1/4*R(-2,1,-1,1) + 1/4*R(-2,0,1,0) + 3/4*R(1,-1,-1,0) + 3/4*R(1,-2,1,-1) + 1/2*R(1,-2,0,1) + 1/2*R(0,1,-2,0) + 9/4*R(0,0,0,-1) + 3/2*R(0,0,-1,1) + 1/2*R(0,-1,2,-2) + 3/2*R(0,-1,1,0) + 1/4*R(-1,2,-1,-1) + 1/4*R(-1,2,-2,1) + 1/4*R(-1,1,1,-2) + 3/4*R(-1,1,0,0) + 1/4*R(-1,0,2,-1) + 1/2*R(2,-1,0,-1) + 1/4*R(2,-1,-1,1) + 1/4*R(2,-2,1,0) + 1/4*R(1,1,-1,-1) + 1/4*R(1,1,-2,1) + 1/4*R(1,0,1,-2) + 3/4*R(1,0,0,0) + 1/4*R(1,-1,2,-1), 33: -1/6*R(-1,-1,0,0) - 1/6*R(-2,1,-1,0) - 1/6*R(-2,0,1,-1) - 1/6*R(1,-2,0,0) - 1/3*R(0,0,-1,0) - 1/3*R(0,-1,1,-1) - 1/6*R(-1,2,-2,0) - 1/3*R(-1,1,0,-1) - 1/6*R(2,-1,-1,0) - 1/6*R(2,-2,1,-1) - 1/6*R(1,1,-2,0) - 1/3*R(1,0,0,-1) - 1/6*R(0,2,-1,-1), 26: 1/12*R(-1,0,-1,-1) + 1/12*R(-1,0,-2,1) + 1/12*R(-1,-1,1,-2) + 1/6*R(-1,-1,0,0) + 1/12*R(-1,-1,-1,2) + 1/12*R(1,-1,-1,-1) + 1/12*R(1,-1,-2,1) + 1/12*R(1,-2,1,-2) + 1/6*R(1,-2,0,0) + 1/12*R(1,-2,-1,2) + 1/12*R(0,0,0,-2) + 1/3*R(0,0,-1,0) + 1/12*R(0,0,-2,2) + 1/4*R(0,-1,1,-1) + 1/4*R(0,-1,0,1), 22: 1/2*R(0,0,-1,0) + 1/2*R(0,-1,1,-1) + 1/2*R(0,-1,0,1), 16: -1/4*R(-1,0,-1,1) - 1/4*R(-1,0,0,-1) - 1/4*R(1,-1,-1,1) - 1/4*R(1,-1,0,-1), 8: -1/4*R(0,-1,0,-1) - 1/4*R(0,-1,-1,1) - 1/4*R(-1,1,-1,-1) - 1/4*R(-1,1,-2,1) - 1/4*R(-1,0,0,0) - 1/4*R(1,0,-1,-1) - 1/4*R(1,0,-2,1) - 1/4*R(1,-1,0,0) - 1/4*R(0,1,-1,0), 3: 1/2*R(-1,0,0,0) + 1/2*R(1,-1,0,0) + 1/2*R(0,1,-1,0), 116: 1/12*R(-1,0,-1,-1) + 1/12*R(-1,-1,1,-2) + 1/6*R(0,0,-1,0) + 1/6*R(0,-1,1,-1) - 1/12*R(-1,1,0,-1), 112: 1/6*R(0,-2,1,-1) + 1/6*R(0,-1,-1,0) + 1/6*R(-1,1,-2,0) + 1/6*R(-1,-1,2,-2), 108: 1/12*R(-1,-1,0,-1) + 1/12*R(-2,1,-1,-1) + 1/6*R(0,-1,0,0) + 1/6*R(-1,1,-1,0) - 1/12*R(-1,0,1,-1), 104: 1/4*R(0,-1,0,-1) + 1/4*R(-1,1,-1,-1) + 1/4*R(-1,0,1,-2), 101: 1/2*R(0,-1,0,0) + 1/2*R(-1,1,-1,0) + 1/2*R(-1,0,1,-1), 98: -1/4*R(0,-1,0,-1) - 1/4*R(-1,1,-1,-1), 95: -1/2*R(-1,0,-1,0) - 1/4*R(-1,-1,1,-1) - 1/4*R(-2,1,0,-1) - 1/4*R(1,-1,-1,0) - 1/4*R(1,-2,1,-1) - 1/4*R(0,1,-2,0) - 1/2*R(0,0,0,-1) - 1/4*R(-1,2,-1,-1), 88: 1/12*R(-1,-1,0,-1) + 1/12*R(-1,-1,-1,1) + 1/12*R(0,-1,0,0), 82: -1/6*R(0,-2,0,1) - 1/6*R(0,-2,1,-1) - 1/6*R(0,-1,-1,0) - 1/6*R(-1,1,-2,0) - 1/3*R(-1,-1,1,0) - 1/6*R(-1,-1,2,-2) - 1/6*R(-1,-1,0,2) - 1/3*R(-1,0,-1,1) - 1/3*R(-1,0,0,-1), 78: -1/12*R(-1,-1,0,-1) - 1/12*R(-1,-1,-1,1) - 1/12*R(-2,1,-1,-1) - 1/12*R(-2,1,-2,1) - 1/12*R(-2,0,0,0) - 1/12*R(0,-1,0,0) - 1/12*R(-1,1,-1,0), 74: -1/4*R(0,-1,0,-1) - 1/4*R(0,-1,-1,1) - 1/4*R(-1,1,-1,-1) - 1/4*R(-1,1,-2,1) - 1/4*R(-1,0,1,-2) - 1/2*R(-1,0,0,0) - 1/4*R(-1,0,-1,2), 70: 1/2*R(0,0,-1,0) + 1/2*R(0,-1,1,-1) + 1/2*R(0,-1,0,1) + 1/2*R(-1,1,0,-1) + 1/2*R(-1,1,-1,1) + 1/2*R(-1,0,1,0), 66: -1/6*R(-1,0,-1,-1) - 1/6*R(-1,0,-2,1) - 1/12*R(-1,-1,1,-2) - 1/6*R(-1,-1,0,0) - 1/12*R(-1,-1,-1,2) - 1/12*R(-2,1,0,-2) - 1/6*R(-2,1,-1,0) - 1/12*R(-2,1,-2,2) - 1/12*R(1,-1,-1,-1) - 1/12*R(1,-1,-2,1) - 1/12*R(1,-2,1,-2) - 1/6*R(1,-2,0,0) - 1/12*R(1,-2,-1,2) - 1/12*R(0,1,-2,-1) - 1/12*R(0,1,-3,1) - 1/4*R(0,0,0,-2) - 1/2*R(0,0,-1,0) - 1/4*R(0,0,-2,2) - 1/4*R(0,-1,1,-1) - 1/4*R(0,-1,0,1) - 1/12*R(-1,2,-1,-2) - 1/6*R(-1,2,-2,0) - 1/12*R(-1,2,-3,2) - 1/4*R(-1,1,0,-1) - 1/4*R(-1,1,-1,1), 63: -3/4*R(0,0,0,0) - 1/4*R(-2,1,0,0) - 1/4*R(-1,-1,1,0) - 1/2*R(-1,0,-1,1) - 1/2*R(-1,0,0,-1) - 1/4*R(1,-2,1,0) - 1/4*R(1,-1,-1,1) - 1/4*R(1,-1,0,-1) - 1/4*R(-1,2,-1,0) - 1/4*R(0,1,-2,1) - 1/4*R(0,1,-1,-1), 56: -1/12*R(-1,0,-1,-1) - 1/12*R(-1,-1,1,-2) - 1/12*R(1,-1,-1,-1) - 1/12*R(1,-2,1,-2) - 1/12*R(0,0,0,-2) - 1/12*R(0,0,-1,0) - 1/12*R(0,-1,1,-1), 53: -1/6*R(-1,-1,0,-1) - 1/12*R(-1,-1,-1,1) - 1/12*R(-1,-2,1,0) - 1/12*R(-2,1,-1,-1) - 1/12*R(-2,1,-2,1) - 1/12*R(-2,0,1,-2) - 1/4*R(-2,0,0,0) - 1/12*R(-2,-1,2,-1) - 1/6*R(1,-2,0,-1) - 1/12*R(1,-2,-1,1) - 1/12*R(1,-3,1,0) - 1/6*R(0,0,-1,-1) - 1/6*R(0,0,-2,1) - 1/6*R(0,-1,1,-2) - 1/2*R(0,-1,0,0) - 1/6*R(0,-2,2,-1) - 1/4*R(-1,1,-1,0) - 1/4*R(-1,0,1,-1) - 1/12*R(2,-1,-1,-1) - 1/12*R(2,-1,-2,1) - 1/12*R(2,-2,1,-2) - 1/4*R(2,-2,0,0) - 1/12*R(2,-3,2,-1) - 1/4*R(1,0,-1,0) - 1/4*R(1,-1,1,-1), 50: -3/4*R(0,0,0,0) - 1/4*R(-1,-1,1,0) - 1/4*R(-1,0,-1,1) - 1/2*R(-1,0,0,-1) - 1/4*R(1,-2,1,0) - 1/4*R(1,-1,-1,1) - 1/2*R(1,-1,0,-1) - 1/4*R(0,1,-2,1) - 1/4*R(0,1,-1,-1) - 1/4*R(0,-1,2,-1) - 1/4*R(0,0,1,-2), 44: -1/2*R(0,-1,0,-1) - 1/4*R(-1,1,-1,-1) - 1/4*R(-1,0,1,-2) - 1/4*R(1,0,-1,-1) - 1/4*R(1,-1,1,-2), 39: -1/6*R(0,0,-1,-1), 35: 1/4*R(-1,0,-1,0) + 1/4*R(-1,-1,1,-1) + 1/4*R(-2,1,0,-1) + 1/4*R(1,-1,-1,0) + 1/4*R(1,-2,1,-1) + 1/4*R(0,1,-2,0) + 3/4*R(0,0,0,-1) + 1/4*R(-1,2,-1,-1) + 1/4*R(2,-1,0,-1) + 1/4*R(1,1,-1,-1), 32: -1/2*R(0,0,-1,0) - 1/2*R(0,-1,1,-1) - 1/2*R(-1,1,0,-1) - 1/2*R(1,0,0,-1), 28: -1/4*R(0,-1,0,-1) - 1/4*R(0,-1,-1,1), 24: -1/6*R(0,0,-1,-1) - 1/6*R(0,0,-2,1) - 1/6*R(0,-1,1,-2) - 1/6*R(0,-1,-1,2), 20: 1/2*R(0,0,0,-1) + 1/2*R(0,0,-1,1) + 1/2*R(0,-1,1,0), 14: 1/2*R(0,-1,0,-1) + 1/2*R(0,-1,-1,1) + 1/4*R(-1,1,-1,-1) + 1/4*R(-1,1,-2,1) + 1/4*R(-1,0,1,-2) + 1/2*R(-1,0,0,0) + 1/4*R(-1,0,-1,2) + 1/4*R(1,0,-1,-1) + 1/4*R(1,0,-2,1) + 1/4*R(1,-1,1,-2) + 1/2*R(1,-1,0,0) + 1/4*R(1,-1,-1,2), 10: -R(0,0,-1,0) - R(0,-1,1,-1) - R(0,-1,0,1) - 1/2*R(-1,1,0,-1) - 1/2*R(-1,1,-1,1) - 1/2*R(-1,0,1,0) - 1/2*R(1,0,0,-1) - 1/2*R(1,0,-1,1) - 1/2*R(1,-1,1,0), 5: -1/2*R(-1,0,-1,0) - 1/4*R(-1,-1,1,-1) - 1/4*R(-1,-1,0,1) - 1/4*R(-2,1,0,-1) - 1/4*R(-2,1,-1,1) - 1/2*R(1,-1,-1,0) - 1/4*R(1,-2,1,-1) - 1/4*R(1,-2,0,1) - 1/2*R(0,1,-2,0) - R(0,0,0,-1) - R(0,0,-1,1) - 1/4*R(0,-1,1,0) - 1/4*R(-1,2,-1,-1) - 1/4*R(-1,2,-2,1) - 1/4*R(-1,1,0,0) - 1/4*R(2,-1,0,-1) - 1/4*R(2,-1,-1,1) - 1/4*R(1,1,-1,-1) - 1/4*R(1,1,-2,1) - 1/4*R(1,0,0,0), 1: -R(0,-1,0,0) - R(-1,1,-1,0) - 1/2*R(-1,0,1,-1) - 1/2*R(-1,0,0,1) - R(1,0,-1,0) - 1/2*R(1,-1,1,-1) - 1/2*R(1,-1,0,1) - 1/2*R(0,1,0,-1) - 1/2*R(0,1,-1,1), 118: -1/12*R(-1,-1,0,-1) - 1/4*R(0,-1,0,0), 115: -1/4*R(-1,0,-1,0) - 1/4*R(-1,-1,1,-1), 111: -1/2*R(0,-1,0,-1) - 1/4*R(0,-1,-1,1) - 1/4*R(0,-2,1,0) - 1/4*R(-1,1,-1,-1) - 1/4*R(-1,1,-2,1) - 1/4*R(-1,0,1,-2) - 1/2*R(-1,0,0,0) - 1/4*R(-1,-1,2,-1), 109: -1/12*R(-1,0,-1,-1) - 1/4*R(0,0,-1,0), 105: 1/4*R(-1,0,-1,0) + 1/4*R(-1,-1,1,-1) + 1/4*R(-2,1,0,-1), 103: -1/6*R(-1,-1,0,0) - 1/6*R(-2,1,-1,0) - 1/6*R(-2,0,1,-1), 100: -1/2*R(-1,0,-1,0) - 1/2*R(-1,-1,1,-1) - 1/4*R(-1,-1,0,1) - 1/2*R(-2,1,0,-1) - 1/4*R(-2,1,-1,1) - 1/4*R(-2,0,1,0) - 1/4*R(1,-1,-1,0) - 1/4*R(1,-2,1,-1) - 1/4*R(1,-2,0,1) - 1/4*R(0,1,-2,0) - R(0,0,0,-1) - 3/4*R(0,0,-1,1) - 1/4*R(0,-1,2,-2) - 3/4*R(0,-1,1,0) - 1/4*R(-1,2,-1,-1) - 1/4*R(-1,2,-2,1) - 1/4*R(-1,1,1,-2) - 3/4*R(-1,1,0,0) - 1/4*R(-1,0,2,-1), 97: 1/6*R(0,-2,1,-1) + 1/6*R(0,-1,-1,0) + 1/6*R(-2,2,-1,-1) + 1/6*R(-1,1,-2,0), 94: -1/6*R(0,0,-1,-1) - 1/6*R(0,-1,1,-2) - 1/6*R(-1,1,0,-2), 92: 1/2*R(0,0,-1,0) + 1/2*R(0,-1,1,-1) + 1/2*R(-1,1,0,-1), 90: R(0,0,0,0) + 1/4*R(-2,1,0,0) + 1/4*R(-1,-1,1,0) + 1/4*R(-1,0,-1,1) + 1/2*R(-1,0,0,-1) + 1/4*R(1,-2,1,0) + 1/4*R(1,-1,-1,1) + 1/4*R(1,-1,0,-1) + 1/4*R(-1,2,-1,0) + 1/4*R(-1,1,1,-1) + 1/4*R(0,1,-2,1) + 1/4*R(0,1,-1,-1) + 1/4*R(0,-1,2,-1) + 1/4*R(0,0,1,-2), 87: -1/6*R(-1,-1,0,0), 85: 1/4*R(-1,0,-1,0) + 1/4*R(-1,-1,1,-1) + 1/4*R(-1,-1,0,1), 83: -1/4*R(-1,-1,1,0) - 1/4*R(-1,0,-1,1) - 1/4*R(-1,0,0,-1), 81: 1/4*R(0,-1,0,-1) + 1/4*R(0,-1,-1,1) + 1/4*R(0,-2,1,0) + 1/4*R(-1,1,-1,-1) + 1/4*R(-1,1,-2,1) + 1/4*R(-1,0,1,-2) + 3/4*R(-1,0,0,0) + 1/4*R(-1,0,-1,2) + 1/4*R(-1,-1,2,-1) + 1/4*R(-1,-1,1,1), 79: 1/4*R(-1,0,-1,0), 77: 1/6*R(-1,-1,0,0) + 1/6*R(-2,1,-1,0), 75: -1/2*R(-1,0,-1,0) - 1/4*R(-1,-1,1,-1) - 1/4*R(-1,-1,0,1) - 1/4*R(-2,1,0,-1) - 1/4*R(-2,1,-1,1), 73: 1/2*R(-1,0,0,0), 71: -1/2*R(0,-1,0,0) - 1/2*R(-1,1,-1,0) - 1/2*R(-1,0,1,-1) - 1/2*R(-1,0,0,1), 69: -1/6*R(0,-1,-1,0) - 1/6*R(-1,1,-2,0), 67: -1/2*R(0,-1,0,0) - 1/2*R(-1,1,-1,0), 64: 1/6*R(0,0,-1,-1) + 1/6*R(0,0,-2,1) + 1/6*R(0,-1,1,-2) + 1/6*R(0,-1,-1,2) + 1/6*R(-1,1,0,-2) + 1/6*R(-1,1,-2,2), 62: -R(0,0,-1,0) - 1/2*R(0,-1,1,-1) - 1/2*R(0,-1,0,1) - 1/2*R(-1,1,0,-1) - 1/2*R(-1,1,-1,1), 60: -1/2*R(0,0,0,-1) - 1/2*R(0,0,-1,1) - 1/2*R(0,-1,1,0) - 1/2*R(-1,1,0,0), 57: -1/6*R(0,-2,1,-1) - 1/6*R(0,-1,-1,0), 54: 1/6*R(0,0,-1,-1) + 1/6*R(0,-1,1,-2), 52: -1/2*R(0,0,-1,0) - 1/2*R(0,-1,1,-1), 49: 1/12*R(-1,0,-1,-1) + 1/12*R(1,-1,-1,-1) + 1/12*R(0,0,-1,0), 45: -1/4*R(-1,0,-1,0) - 1/4*R(-1,-1,1,-1) - 1/4*R(-2,1,0,-1) - 1/4*R(1,-1,-1,0) - 1/4*R(1,-2,1,-1) - 1/2*R(0,0,0,-1) - 1/4*R(2,-1,0,-1), 43: 1/6*R(-1,-1,0,0) + 1/6*R(-2,1,-1,0) + 1/6*R(-2,0,1,-1) + 1/6*R(1,-2,0,0) + 1/6*R(2,-1,-1,0) + 1/6*R(2,-2,1,-1), 41: -R(0,-1,0,0) - 1/2*R(-1,1,-1,0) - 1/2*R(-1,0,1,-1) - 1/2*R(1,0,-1,0) - 1/2*R(1,-1,1,-1), 38: 1/4*R(0,-1,0,-1) + 1/4*R(-1,1,-1,-1) + 1/4*R(1,0,-1,-1), 36: -1/4*R(-1,0,0,-1) - 1/4*R(1,-1,0,-1) - 1/4*R(0,1,-1,-1), 34: 1/2*R(0,0,0,-1), 31: 1/2*R(0,-1,0,0) + 1/2*R(-1,1,-1,0) + 1/2*R(-1,0,1,-1) + 1/2*R(1,0,-1,0) + 1/2*R(1,-1,1,-1) + 1/2*R(0,1,0,-1), 29: 1/6*R(0,-1,-1,0), 27: 1/2*R(0,-1,0,0), 23: 1/4*R(0,0,0,0) + 1/4*R(-1,-1,1,0) + 1/4*R(-1,0,-1,1) + 1/4*R(-1,0,0,-1) + 1/4*R(1,-2,1,0) + 1/4*R(1,-1,-1,1) + 1/4*R(1,-1,0,-1), 21: -1/2*R(0,-1,0,-1) - 1/2*R(0,-1,-1,1) - 1/2*R(0,-2,1,0) - 1/4*R(-1,1,-1,-1) - 1/4*R(-1,1,-2,1) - 1/4*R(-1,0,1,-2) - R(-1,0,0,0) - 1/4*R(-1,0,-1,2) - 1/4*R(-1,-1,2,-1) - 1/4*R(-1,-1,1,1) - 1/4*R(1,0,-1,-1) - 1/4*R(1,0,-2,1) - 1/4*R(1,-1,1,-2) - R(1,-1,0,0) - 1/4*R(1,-1,-1,2) - 1/4*R(1,-2,2,-1) - 1/4*R(1,-2,1,1) - 1/4*R(0,1,-1,0) - 1/4*R(0,0,1,-1) - 1/4*R(0,0,0,1), 19: -1/4*R(-1,0,-1,0) - 1/4*R(1,-1,-1,0), 17: -1/6*R(-1,-1,0,0) - 1/6*R(-2,1,-1,0) - 1/6*R(1,-2,0,0) - 1/6*R(2,-1,-1,0), 15: 1/2*R(-1,0,-1,0) + 1/4*R(-1,-1,1,-1) + 1/4*R(-1,-1,0,1) + 1/4*R(-2,1,0,-1) + 1/4*R(-2,1,-1,1) + 1/2*R(1,-1,-1,0) + 1/4*R(1,-2,1,-1) + 1/4*R(1,-2,0,1) + 1/2*R(0,0,0,-1) + 1/2*R(0,0,-1,1) + 1/4*R(2,-1,0,-1) + 1/4*R(2,-1,-1,1), 13: -1/2*R(-1,0,0,0) - 1/2*R(1,-1,0,0), 11: R(0,-1,0,0) + 1/2*R(-1,1,-1,0) + 1/2*R(-1,0,1,-1) + 1/2*R(-1,0,0,1) + 1/2*R(1,0,-1,0) + 1/2*R(1,-1,1,-1) + 1/2*R(1,-1,0,1), 9: 1/2*R(0,0,-1,0), 6: 1/4*R(0,0,0,0) + 1/4*R(-1,0,-1,1) + 1/4*R(-1,0,0,-1) + 1/4*R(1,-1,-1,1) + 1/4*R(1,-1,0,-1) + 1/4*R(0,1,-2,1) + 1/4*R(0,1,-1,-1), 4: -1/2*R(0,0,0,-1) - 1/2*R(0,0,-1,1), 2: R(0,0,-1,0) + 1/2*R(0,-1,1,-1) + 1/2*R(0,-1,0,1) + 1/2*R(-1,1,0,-1) + 1/2*R(-1,1,-1,1) + 1/2*R(1,0,0,-1) + 1/2*R(1,0,-1,1), 0: R(0,0,0,0)}

In [4]:
#Defining [q]^*f_w
q = 29

d_qsub = {}
for i in range(len(W)):
    print(i + 1, "out of", len(W), end='\r')
    d_qsub[i] = qfsub(W[i])

120 out of 120

In [5]:
#DEPENDS ON THE TYPE! Set the minimal Duflo involution in each left cell. Leave lcell_dict = {} in Type A!
# B3 EXAMPLE lcell_dict = {s1^2 : s1^2, s1 : s1, s1*s2*s3*s2*s1 : s1,
#              s2 : s2, s2*s3*s2 : s2,
##              s3 : s3, s3*s2*s3 : s3,
#              s3*s1 : s3*s1, s2*s3*s1*s2 : s2*s3*s1*s2, s3*s2*s3*s1*s2*s3 : s3*s2*s3*s1*s2*s3,
#              s1*s2*s1 : s1*s2*s1, s3*s1*s2*s3*s1 : s3*s1*s2*s3*s1, s2*s3*s1*s2*s3*s1*s2 : s2*s3*s1*s2*s3*s1*s2,
#              s3*s2*s3*s2 : s3*s2*s3*s2, s3*s2*s3*s1*s2*s3*s1*s2 : s3*s2*s3*s2,
#              s3*s1*s2*s3*s2*s1 : s3*s1*s2*s3*s2*s1, s3*s2*s3*s1*s2*s3*s2*s1 : s3*s1*s2*s3*s2*s1,
#              s1*s2*s3*s1*s2*s1 : s1*s2*s3*s1*s2*s1, s2*s3*s1*s2*s3*s1*s2*s1 : s1*s2*s3*s1*s2*s1,
#              w0 : w0}
lcell_dict = {}
for a in W:
    if a not in lcell_dict:
        lcell_dict[a] = a
print("lcell_dict is defined.")

lcell_dict is defined.


In [6]:
def nqpair(w, y):
    #new/normalized q_pairing
    return woke_pairing(d_qsub[list(W).index(w)], d_dual[list(W).index(lcell_dict[y])])

table = []

Mw = {}

for w in invols:
    table.append([invols.index(w), str(w), str(nqpair(w0*w, w0*w))])

for row in table:
    print('{:20}| {:40} | {:80}'.format(*row))

                   0| 1                                        | A4(0,0,0,0)                                                                     
                   1| s3                                       | A4(0,0,25,1) - A4(1,0,26,0) + A4(0,0,28,0)                                      
                   2| s3*s4*s2*s3                              | A4(1,0,26,0) + A4(0,0,28,0)                                                     
                   3| s2                                       | A4(1,25,0,0) - A4(0,26,0,1) + A4(0,28,0,0)                                      
                   4| s2*s3*s2                                 | -A4(0,27,27,0) + A4(0,28,28,0)                                                  
                   5| s2*s3*s1*s2                              | A4(0,26,0,1) + A4(0,28,0,0)                                                     
                   6| s2*s3*s4*s1*s2*s3*s1*s2                  | A4(0,27,27,0) + A4(0,28,28,0)                              

In [7]:
table = []
totals = 0

for w in W:
    table.append([list(W).index(w), str(w), str(nqpair(w0*w, w0*w))])
    totals += nqpair(w0*w, w0*w)

for row in table:
    print('{:20}| {:40} | {:80}'.format(*row))

                   0| 1                                        | A4(0,0,0,0)                                                                     
                   1| s3                                       | A4(0,0,25,1) - A4(1,0,26,0) + A4(0,0,28,0)                                      
                   2| s3*s2                                    | A4(1,25,0,0) - A4(0,26,0,1) + A4(0,28,0,0)                                      
                   3| s3*s4                                    | A4(0,0,0,28)                                                                    
                   4| s3*s2*s1                                 | A4(28,0,0,0)                                                                    
                   5| s3*s4*s2                                 | A4(0,26,1,27) - A4(0,26,0,29) - A4(1,27,0,27) + A4(0,28,0,28)                   
                   6| s3*s4*s2*s1                              | -A4(26,0,0,26) + A4(28,0,0,28)                             

In [20]:
table = []
totals = 0

for w in [s2*s3*s2, s1*s2*s3*s4*s3*s2*s1]:
    table.append([list(W).index(w), str(w), str(nqpair(w0*w, w0*w))])
    totals += nqpair(w0*w, w0*w)

for row in table:
    print('{:20}| {:40} | {:80}'.format(*row))

print()

table = []
table.append([list(W).index(s2*s3*s2), str(s2*s3*s2), str("-A4(0,27,27,0) + A4(27,0,0,27) + A4(0,28,28,0)")])
table.append([list(W).index(s1*s2*s3*s4*s3*s2*s1), str(s1*s2*s3*s4*s3*s2*s1), str("A4(26,0,0,26) + A4(27,0,0,27) + A4(28,0,0,28)")])

for row in table:
    print('{:20}| {:40} | {:80}'.format(*row))

                  12| s2*s3*s2                                 | -A4(0,27,27,0) + A4(0,28,28,0)                                                  
                 106| s1*s2*s3*s4*s3*s2*s1                     | A4(26,0,0,26) + 2*A4(27,0,0,27) + A4(28,0,0,28)                                 

                  12| s2*s3*s2                                 | -A4(0,27,27,0) + A4(27,0,0,27) + A4(0,28,28,0)                                  
                 106| s1*s2*s3*s4*s3*s2*s1                     | A4(26,0,0,26) + A4(27,0,0,27) + A4(28,0,0,28)                                   
